# **Plant Disease**

In [ ]:
# importing libraries
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import pandas
from tensorflow import keras
from keras.preprocessing.image import ImageDataGenerator as Imgen ,image_dataset_from_directory
from keras.models import Model,Sequential
from keras.layers import Input,Conv2D,MaxPooling2D,Dropout,Flatten,Dense,GlobalAveragePooling2D,BatchNormalization
from keras.callbacks import EarlyStopping,ModelCheckpoint

from keras.applications.mobilenet_v2 import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input

from sklearn.metrics import confusion_matrix,classification_report
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from PIL import Image
import os
import pickle
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Flatten, Dense, Conv2D, MaxPool2D, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelBinarizer
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix
import itertools
from keras.preprocessing.image import image_dataset_from_directory

In [ ]:

# Dimension of resized image
DEFAULT_IMAGE_SIZE = tuple((32, 32))

# Path Cwd
cwd_dir = os.getcwd()

print(cwd_dir)
# Path to the dataset folder
root_dir = cwd_dir+'/Dataset'

train_dir = os.path.join(root_dir, 'train').replace("\\", '/')
val_dir = os.path.join(root_dir, 'val').replace("\\", '/')
test_dir = os.path.join(root_dir, 'test').replace("\\", '/')
print(test_dir)

BATCH_SIZE = 32
CHANNELS=3
EPOCHS=5

### **Data Generation**

In [ ]:
traingen = Imgen(preprocessing_function=preprocess_input)
valgen = Imgen(preprocessing_function=preprocess_input)
testgen = Imgen(preprocessing_function=preprocess_input)

In [ ]:
trainds = traingen.flow_from_directory(train_dir,
                                      target_size=DEFAULT_IMAGE_SIZE,
                                       class_mode="categorical",
                                       seed=123,
                                       batch_size=32,
                                       subset="training"
                                      )

testds = testgen.flow_from_directory(val_dir,target_size=DEFAULT_IMAGE_SIZE,
                                    class_mode="categorical",
                                    seed=123,
                                    batch_size=32,
                                    shuffle=False)

valds = valgen.flow_from_directory(test_dir,
                                      target_size=DEFAULT_IMAGE_SIZE,
                                       class_mode="categorical",
                                       seed=123,
                                       batch_size=32
                                       
                                      )

**Classes**

In [ ]:
c = trainds.class_indices
classes = list(c.keys())
classes

In [ ]:
plt.figure(figsize=[20,8])
sns.countplot(x=trainds.classes)

**One Batch**

In [ ]:
x,y = next(trainds)

# function to plot images
def plotImages(x,y):
    plt.figure(figsize=[15,11])
    for i in range(16):
        plt.subplot(4,4,i+1)
        plt.imshow(x[i])
        plt.title(classes[np.argmax(y[i])])
        plt.axis("off")
    plt.show()

In [ ]:
plotImages(x,y)

## **Model**

In [ ]:
WIDTH = DEFAULT_IMAGE_SIZE[0]
HEIGHT = DEFAULT_IMAGE_SIZE[1]
DEPTH = 3
inputShape = (HEIGHT, WIDTH, DEPTH)

In [ ]:

## Tetsing layer

model = Sequential()
# Extracted Feature Layer
model.add(InputLayer(input_shape=inputShape))
model.add(Conv2D(filters=32, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(Conv2D(filters=32, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(MaxPool2D(pool_size=(2, 2), padding='same'))
model.add(Dropout(rate=0.25))
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(MaxPool2D(pool_size=(2, 2), padding='same'))
model.add(Dropout(0.25))
model.add(Flatten())
# Fully Connected Layer
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(19, activation='softmax'))

In [ ]:
model.summary()

**Compile**

In [ ]:
# tentukan hyperparameter
lr = 0.001
max_epochs = 20 #100
opt_funct = Adam(learning_rate=lr)

In [ ]:
#Compilation of the model
model.compile(loss='binary_crossentropy', optimizer=opt_funct, metrics=['accuracy'])

**Train**

In [ ]:
hist = model.fit(trainds,batch_size = 32,epochs=max_epochs,validation_data=valds)

**Test**

In [ ]:
model.evaluate(testds)

In [ ]:
plt.figure(figsize=(15,6))

plt.subplot(1,2,1)
plt.plot(hist.epoch,hist.history['accuracy'],label = 'Training')
plt.plot(hist.epoch,hist.history['val_accuracy'],label = 'validation')

plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(hist.epoch,hist.history['loss'],label = 'Training')
plt.plot(hist.epoch,hist.history['val_loss'],label = 'validation')

plt.title("Loss")
plt.legend()
plt.show()

## **Predictions.**

In [ ]:
c = trainds.class_indices
classes = list(c.keys())
c

In [ ]:
pred = model.predict(testds)

In [ ]:
pred = [np.argmax(i) for i in pred]

In [ ]:
y_test = testds.classes
print(y_test)

**Classification Report**

In [ ]:
print(classification_report(pred,y_test))
report = classification_report(pred,y_test)
df = pandas.DataFrame(report).transpose()
df.to_csv('classification_report.csv', index = False)


**Confusion Matrix**

In [ ]:
plt.figure(figsize=[20,15])
sns.heatmap(confusion_matrix(pred,y_test),annot = True, fmt = "d", cmap = "viridis")